# MARKET PULSE · Colab 분석 노트북 (개정판)

**가격·거래량 / 기술적 지표 / 뉴스 감성(가상) / 다음 거래일 확률 모델 / 백테스트**를 설명하고 시연합니다.

웹앱 UI는 `app.py`를 PC에서 Streamlit으로 실행해야 합니다. 이 노트북은 인터넷 없이 **가상 OHLCV 및 가상 뉴스 데이터**를 분석할 수 있도록 만든 별도 실습 환경입니다. 실제 주가나 뉴스가 아닙니다. 여기서 산출하는 확률은 통계 모델의 미보정 추정치이며 매매 추천이 아닙니다.

> 이 노트북 사용 전 `MarketPulse_KR_개정판.zip`을 Colab에 업로드하거나 동일한 프로젝트 폴더에 노트북을 두세요.

## 1. 필요한 패키지 설치

Colab에서는 아래 셀을 실행합니다. 일반 Python 파일에서는 `!pip` 문법을 사용하지 마세요.

In [ ]:
# Colab cell
%pip -q install "pandas>=2,<3" "numpy>=1.24,<3" "plotly>=5.18,<7" "scikit-learn>=1.3,<2"

## 2. 프로젝트 압축파일 업로드 및 모듈 불러오기

Colab에서 실행한다면 압축파일을 업로드해 주세요. 일반 Jupyter에서 프로젝트 폴더 안에서 열었다면 업로드를 생략합니다.

In [ ]:
from pathlib import Path
import sys, zipfile
root = Path.cwd()
if not (root / "stock_core.py").exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        archive = next((n for n in uploaded if n.lower().endswith('.zip')), None)
        if archive is None:
            raise RuntimeError('프로젝트 ZIP을 업로드해야 합니다.')
        with zipfile.ZipFile(archive) as zf:
            # A local user-supplied ZIP with normal file paths is expected.
            safe_files = [p for p in zf.namelist() if not Path(p).is_absolute() and '..' not in Path(p).parts]
            if len(safe_files) != len(zf.namelist()):
                raise RuntimeError('안전하지 않은 ZIP 경로가 있습니다.')
            zf.extractall(root)
        possible = list(root.rglob('stock_core.py'))
        if not possible: raise RuntimeError('ZIP에서 stock_core.py를 찾지 못했습니다.')
        root = possible[0].parent
    except ImportError:
        raise RuntimeError('프로젝트 루트에서 이 노트북을 실행하거나 Colab에 ZIP을 업로드하세요.')
sys.path.insert(0, str(root))
print('프로젝트 폴더:', root)

## 3. 가상 기업의 과거 가격·거래량 데이터 확인

가상기업 A~D는 난수로 생성한 시연용 가격 데이터입니다. 실제 시장을 재현하지 않습니다.

In [ ]:
from stock_core import make_demo_data, calculate_indicators, event_table, summarize, trim_period
from technical_indicators import add_indicators
from datetime import date
import pandas as pd

demo = make_demo_data()
for name, frame in demo.items():
    print(name, len(frame), frame['Date'].min().date(), frame['Date'].max().date())
stock = '가상기업 A'
frame = demo[stock]
indicators = add_indicators(frame)
indicators[['Date', 'Close', 'Volume', 'RSI_14', 'MACD', 'Volatility_20d']].tail()

## 4. 통계 급변 탐지

`Z = |당일 수익률 - 직전 N거래일 평균| / 직전 N거래일 표준편차`를 계산합니다. **당일을 평가하는 기준에는 당일 값이 포함되지 않습니다.** 탐지는 이미 일어난 이상 움직임을 설명하며 미래 가격 예측과 다릅니다.

In [ ]:
results={}
for name, raw in demo.items():
    calculated=calculate_indicators(raw,window=20,z_limit=2.5,q_limit=1.5)
    results[name]=trim_period(calculated, date(2025,1,1),date(2026,11,6))
comparison=summarize(results)
print('종목별 변동성 및 탐지 횟수')
display(comparison)
detected=event_table(results)
display(detected.head(10))

## 5. 차트 시각화

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
fig=make_subplots(rows=2,cols=1,shared_xaxes=True,row_heights=[.75,.25])
fig.add_trace(go.Scatter(x=frame['Date'],y=frame['Close'],name='가상 종가'),row=1,col=1)
fig.add_trace(go.Bar(x=frame['Date'],y=frame['Volume'],name='가상 거래량'),row=2,col=1)
fig.update_layout(title=f'{stock} (실제 주가 아님)',height=650)
fig.show()

## 6. 합성 금융 뉴스 키워드 감성 분석

가상 뉴스의 `first_seen_at`은 합성 생성 시점으로 만들어진 교육용 데이터입니다. 실제 뉴스 API에서 오늘 처음 조회한 옛 기사는 과거 날짜의 정보로 소급해서 사용하지 않습니다.

In [ ]:
from news_collector import read_news_csv
from news_analyzer import analyze_articles
news_all = read_news_csv((root/'sample_data'/'가상뉴스_시연.csv').read_bytes(), synthetic=True)
news = news_all[news_all['stock_code'] == stock].reset_index(drop=True)
news_scored=analyze_articles(news)
print('가상 뉴스 건수:',len(news_scored))
display(news_scored[['published_at','title','sentiment_label','sentiment_score','evidence']].head())

## 7. 모델 A/B/C 비교와 테스트 기간의 성능 측정

모델 A: 가격·거래량, 모델 B: +기술적 지표, 모델 C: +뉴스 감성. 60/20/20 시간순 분리와 경계 간격을 사용합니다. 검증 데이터의 미래 정보 누출을 방지하며, **가상 예측 성능을 실제 시장 성능이라고 주장하지 않습니다.**

In [ ]:
from prediction_model import run_experiments
report=run_experiments(frame,news=news)
display(report.summary)
display(report.forecasts)
print(report.news_note)

## 8. Walk-forward와 평가 결과

최종 시험 구간의 점수를 먼저 확인하고, 훈련 구간을 앞으로 이동하는 추가 평가를 실시합니다. Brier Score는 낮을수록 좋으며, 예측 모델이 기준 모델보다 성능이 떨어질 수도 있습니다.

In [ ]:
from feature_engineering import usable_history
from backtesting import walkforward_scores
model_name='가격·거래량'
ev=report.evaluations[model_name]
display(ev.prediction_rows.head())
history=usable_history(report.history,ev.feature_names)
display(walkforward_scores(history,ev.feature_names,splits=3))

## 9. 데이터 및 실험 결과 내보내기

In [ ]:
comparison.to_csv('market_pulse_comparison.csv',index=False,encoding='utf-8-sig')
detected.to_csv('market_pulse_anomalies.csv',index=False,encoding='utf-8-sig')
report.summary.to_csv('market_pulse_model_scores.csv',index=False,encoding='utf-8-sig')
report.forecasts.to_csv('market_pulse_predictions.csv',index=False,encoding='utf-8-sig')
print('완료: 4개 CSV 결과 파일 저장')

## 10. 실제 주가 조회 안내 및 제한사항

실제 데이터를 조회하려면 별도로 프로젝트에서 `python collect_real_data.py`를 실행하거나 웹앱에서 **국내 주식 실데이터** 모드를 선택합니다. FinanceDataReader는 외부 제공처에 의존하며, 접속 실패 시 로컬의 이전 CSV로 분석할 수 있으나 **최신이라고 표시하지 않습니다**.

뉴스 검색은 NAVER API 키, OpenDART 공시는 별도의 DART API 키가 필요합니다. 본 Colab 실습은 인증키 없이도 완료할 수 있게 가상 데이터만 사용합니다. 기사 저작권·인증키 보호를 준수하고 예측값을 매매 지시로 사용하지 마세요.